## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os

from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
  result = await Runner.run(search_agent, message)
  
display(Markdown(f"**Summary:** {result.final_output}"))

**Summary:** In 2025, several AI agent frameworks have emerged, each catering to specific needs in the AI development landscape. LangChain remains the most popular, offering a comprehensive ecosystem for building large language model (LLM) applications, particularly excelling in retrieval-augmented generation (RAG) and document processing tasks. CrewAI has gained traction for its role-based approach to multi-agent collaboration, making it ideal for content generation and research automation. Microsoft's AutoGen stands out in conversational multi-agent systems, especially for code generation and research automation scenarios. ([agentframeworkhub.com](https://www.agentframeworkhub.com/blog/best-ai-agent-frameworks-2025?utm_source=openai))

OpenAI's Agents SDK has become a benchmark for production-grade AI agent frameworks, emphasizing reliability and deep model integration. Google's Agent Development Kit (ADK) offers a modular and cloud-native approach, facilitating scalable AI agent deployment. LangGraph, built upon LangChain, introduces stateful, graph-based orchestration, enhancing control over complex workflows. CrewAI and PydanticAI focus on role-based multi-agent collaboration, catering to specialized tasks. Temporal provides a reliability backbone for AI agent orchestration, ensuring operational stability. ([devnavigator.com](https://devnavigator.com/2025/11/20/the-state-of-ai-agent-frameworks-in-2025/?utm_source=openai))

Additionally, Nvidia's Nemotron Coalition, comprising eight AI labs, is co-developing open frontier models on NVIDIA DGX Cloud, supporting the development of Nvidia’s upcoming Nemotron 4 model family. This initiative aims to advance multimodal technology, coding benchmarks, and long-horizon reasoning capabilities. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai))

For a more in-depth comparison and practical guide to these frameworks, you might find the following video helpful:

[Top 5 AI Agent Frameworks in 2025: A Practical Guide for AI Builders](https://www.youtube.com/watch?v=jb4-1MhnGv8&utm_source=openai)
 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [5]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="The reason for performing this search, what you hope to find")
    query: str = Field(description="The search term to query for")
    
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description=f"A list of {HOW_MANY_SEARCHES} search items, each with a reason and query")
    
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan
)

In [6]:
message = "Latest AI agent frameworks in 2025"

with trace("Search"):
  result = await Runner.run(planner_agent, message)
  print(result.final_output)

searches=[WebSearchItem(reason='To find the most recent AI agent frameworks released in 2025', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To get insights on the features and updates of AI agent frameworks released in 2025', query='AI agent frameworks features 2025'), WebSearchItem(reason='To explore comparisons between different AI agent frameworks available in 2025', query='compare AI agent frameworks 2025')]


In [17]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tangotew@gmail.com") # Change this to your verified email
    to_email = To("tangogatdet76@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [18]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [19]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

# Let's create Report structure output schema
class ReportData(BaseModel):
    short_summary: str = Field(description="A concise summary of the report, no more than 3 sentences")
    
    markdown_report: str = Field(description="The full report in markdown format, with appropriate sections and formatting")
    
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")
    
# create the writter agent
writer_agent = Agent(
  name="WritterAgent",
  instructions=INSTRUCTIONS,
  model="gpt-4o-mini",
  output_type=ReportData
)

### Create Runner functions that will plan and execute the search, using planner and search agent

In [20]:
async def plan_research(query: str) -> WebSearchPlan:
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches.")
    return result.final_output
  
async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan"""
    input = f"Search term: {item.query}\nReason for search: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output
  
async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    return results

In [24]:
# write a report agent runner agent function
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    return result.final_output
  
async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report"""
    print("Writing email...")
    print(f"Email subject: {report.short_summary}")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report 

In [25]:
await Runner.run(email_agent, "# Report\n\nThis is the full report in markdown format.",)

RunResult(input='# Report\n\nThis is the full report in markdown format.', new_items=[MessageOutputItem(agent=Agent(name='Email agent', handoff_description=None, tools=[FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10ad7b240>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)], mcp_servers=[], mcp_config={}, instructions='You are able to send a nicely formatted HTML email based on a detailed report.\nYou will be provided with a detailed report. You should use your tool to send one email, providing the \nreport converted into clean,

In [26]:
await send_email(ReportData(
    short_summary="This is a summary of the report.",
    markdown_report="# Report\n\nThis is the full report in markdown format.",
    follow_up_questions=["What are the ethical implications of AI agents?", "How do AI agents impact the job market?"]
))

Writing email...
Email subject: This is a summary of the report.
Email sent


ReportData(short_summary='This is a summary of the report.', markdown_report='# Report\n\nThis is the full report in markdown format.', follow_up_questions=['What are the ethical implications of AI agents?', 'How do AI agents impact the job market?'])

### Now Create the pipeline to run the search and send the report using email agent

In [27]:
query = "Latest AI agent frameworks in 2025"

with trace("Research trace"):
    print("Starting the research...")
    search_plan = await plan_research(query) # get the search plan
    search_results = await perform_searches(search_plan) # perform the searches
    report: ReportData = await write_report(query, search_results) # write the report
    await send_email(report) # send the report via email
    print("Research complete!")
    

Starting the research...
Planning searches...
Will perform 3 searches.
Searching...
Thinking about report...
Writing email...
Email subject: In 2025, notable AI agent frameworks such as OpenAI's Codex, Google's Antigravity, and IBM's BeeAI emerged, significantly advancing intelligent system development and deployment. These frameworks enhance automation across industries while addressing security concerns associated with AI proliferation. The landscape reflects a diversity of tools designed for various applications, underscoring the importance of selecting a framework tailored to specific organizational goals.
Email sent
Research complete!
